# Hướng Dẫn Chạy Pipeline Sinh Dữ Liệu Fake News Đa Phương Thức Trên Google Colab GPU

Notebook này hướng dẫn chi tiết các bước để thiết lập môi trường, nạp mô hình Qwen2.5-7B-Instruct và Gemma-3-4B-it bằng 4-bit BitsAndBytes, và chạy pipeline sinh dữ liệu Fake News đa phương thức.

## 1. Kiểm tra GPU
Đảm bảo bạn đã chọn Runtime -> Change runtime type -> **T4 GPU** (hoặc A100 / L4 nếu có Colab Pro).

In [ ]:
!nvidia-smi

## 2. Kết nối Google Drive và tải mã nguồn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Chuyển vào thư mục dự án trên Google Drive
%cd /content/drive/MyDrive/real_data

## 3. Cài đặt các thư viện cần thiết

In [ ]:
!pip install -q -r requirements.txt
!pip install -q bitsandbytes accelerate

## 4. Chạy kiểm thử nhanh ở chế độ Dry-Run (5 - 15 mẫu)

In [ ]:
# Chạy unit tests toàn diện
!python run_dry_run_test.py

# Chạy thử nghiệm 15 mẫu không cần tải mô hình lớn
!python main.py --dry-run --limit 15 --no-resume

## 5. Chạy toàn bộ Pipeline với LLM và VLM (GPU 4-bit Quantization)
Cấu hình trong `configs/config.yaml` đã được tối ưu cho Colab:
- `load_in_4bit: true`
- `sequential_execution: true` (Giải phóng VRAM giữa LLM và VLM để không bao giờ bị OOM)

In [ ]:
# Chạy 100 bài Good đầu tiên (Giai đoạn 1 trong plan.md)
!python main.py --limit 100

# Hoặc chạy toàn bộ tập dữ liệu (có hỗ trợ resume tự động khi bị ngắt kết nối)
# !python main.py

## 6. Kiểm tra kết quả dữ liệu xuất ra

In [ ]:
import json

# Đọc và hiển thị tóm tắt thống kê
with open('outputs/final_dataset/dataset_summary.json', 'r', encoding='utf-8') as f:
    summary = json.load(f)
print(json.dumps(summary, indent=2, ensure_ascii=False))

# Xem thử 2 mẫu đầu tiên trong train.jsonl
with open('outputs/final_dataset/train.jsonl', 'r', encoding='utf-8') as f:
    lines = [json.loads(next(f)) for _ in range(2)]
print('\nVí dụ mẫu dữ liệu xuất:')
print(json.dumps(lines, indent=2, ensure_ascii=False))